# Level 4 — Baseline klasifikasi Emergency / Non-Emergency (Tahap E proyek)

> **Data sintetis.** Skor di notebook ini tidak membuktikan kelayakan model untuk tanggap darurat nyata
> (lihat `../disaster-classification/DATA_CARD.md`).

Tiga baseline (TF-IDF + Multinomial NB / Logistic Regression / Linear SVM) dibandingkan dengan **data,
pembagian per kejadian, seed, dan metrik yang sama**. Prasyarat: `pip install -r requirements-ml.txt`.

In [1]:
import sys
import warnings
from pathlib import Path

# Notebook ini berada di examples/<folder-level>/. Paket nlp_hub ada di examples/.
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore", message="'count' is passed as positional argument")

## 1. Data dan pembagian per kejadian
Duplikat persis dibuang. Tidak ada `event_id` yang muncul di lebih dari satu bagian.

In [2]:
import pandas as pd
from nlp_hub.data import bagi_latih_val_uji, kodekan_label, muat_dataset, muat_uji_tangan

df = muat_dataset()
latih, val, uji = bagi_latih_val_uji(df)
dev = pd.concat([latih, val], ignore_index=True)   # data pengembangan
tangan = muat_uji_tangan()
print(len(latih), len(val), len(uji), "| tulis-tangan:", len(tangan))
print("kejadian tumpang tindih:", len(set(latih.event_id) & set(uji.event_id)))

335 117 112 | tulis-tangan: 48
kejadian tumpang tindih: 0


## 2. Tiga baseline dengan pipeline
`buat_pipeline` merangkai preprocessing → TF-IDF → model. `fit` hanya melihat data latih.

In [3]:
from nlp_hub.modeling import buat_pipeline

y_dev = kodekan_label(dev["label"])
model = {n: buat_pipeline(k).fit(dev["text"], y_dev) for n, k in [("Naive Bayes", "nb"), ("Logistic Regression", "lr"), ("Linear SVM", "svm")]}
list(model)

['Naive Bayes', 'Logistic Regression', 'Linear SVM']

## 3. Evaluasi: validasi silang berkelompok + uji sekali
F1 makro dan recall kelas Emergency; Emergency = 1 (kelas positif).

In [4]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from nlp_hub.eksperimen import metrik_emergency

X, y = dev["text"].to_numpy(), y_dev.to_numpy()
baris = {}
for nama, kode_algo in [("Naive Bayes", "nb"), ("Logistic Regression", "lr"), ("Linear SVM", "svm")]:
    hasil = []
    for i_latih, i_uji in StratifiedGroupKFold(5, shuffle=True, random_state=42).split(X, y, groups=dev["event_id"]):
        m = buat_pipeline(kode_algo).fit(X[i_latih], y[i_latih])
        hasil.append(metrik_emergency(y[i_uji], m.predict(X[i_uji])))
    baris[nama] = {"F1 makro CV": round(np.mean([h["f1_makro"] for h in hasil]), 3), "Recall E CV": round(np.mean([h["recall_e"] for h in hasil]), 3)}
pd.DataFrame(baris).T

,F1 makro CV,Recall E CV
Naive Bayes,0.983,0.994
Logistic Regression,0.975,0.941
Linear SVM,0.995,0.994


In [5]:
y_uji, y_tan = kodekan_label(uji["label"]), kodekan_label(tangan["label"])
tab = {}
for nama, m in model.items():
    a, b = metrik_emergency(y_uji, m.predict(uji["text"])), metrik_emergency(y_tan, m.predict(tangan["text"]))
    tab[nama] = {"F1 makro uji": round(a["f1_makro"], 3), "Recall E uji": round(a["recall_e"], 3),
                 "F1 makro tangan": round(b["f1_makro"], 3), "Recall E tangan": round(b["recall_e"], 3)}
pd.DataFrame(tab).T

,F1 makro uji,Recall E uji,F1 makro tangan,Recall E tangan
Naive Bayes,0.97,0.974,0.916,0.833
Logistic Regression,0.98,0.947,0.681,0.417
Linear SVM,0.98,1.000,0.806,0.625


**Membaca hasil:** bandingkan kolom uji sintetis dengan tulis-tangan. Selisih besar menunjukkan model
mengenali gaya templat, bukan memahami bahasa. Peringkat dapat berbeda antar-kolom; jangan menyebut satu model terbaik.

## 4. Memprediksi teks baru dengan model Logistic Regression

In [6]:
baru = ["Tolong, air sudah masuk rumah kami di Jl. Melati RT 04, ada lansia yang tidak bisa jalan",
        "Musim hujan sudah dekat, waspada banjir ya semuanya"]
for t, p in zip(baru, model["Logistic Regression"].predict_proba(baru)[:, 1]):
    print(f"P(Emergency)={p:.3f} | {t[:60]}")

P(Emergency)=0.753 | Tolong, air sudah masuk rumah kami di Jl. Melati RT 04, ada 
P(Emergency)=0.093 | Musim hujan sudah dekat, waspada banjir ya semuanya


Probabilitas ini belum terkalibrasi dan ambang 0,5 hanya bawaan. Keputusan operasional memerlukan evaluasi
(Level 5) dan peninjauan manusia.

## Latihan
1. Tambahkan `buat_pipeline("lr", C=10)` ke sel 3 dan bandingkan recall Emergency tulis-tangan. Apakah selisihnya lebih besar daripada satu teks (0,042)?
2. Cetak confusion matrix untuk SVM pada uji sintetis dan pada tulis-tangan (Level 5).